In [3]:
import os
import whisper
import torch  # 新增：用于检测 GPU 状态

In [9]:
_MODEL = None
# 自动检测设备：如果有 CUDA 则用 GPU，否则回退到 CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def get_model():
    global _MODEL
    if _MODEL is None:
        # 修改点 1：使用 .en 后缀的专用模型，性能比通用模型更好
        print(f"正在加载英语专用模型至设备: {DEVICE}...")
        _MODEL = whisper.load_model("small.en", device=DEVICE) 
    return _MODEL

def has_voice_whisper(file_path: str, min_chars: int = 8) -> bool:
    model = get_model()
    use_fp16 = True if DEVICE == "cuda" else False
    
    # 直接转写，不要进行语种检测
    # 因为使用 small.en 模型，它只能识别英语
    result = model.transcribe(
        file_path, 
        language="en",  # 必须显式指定或保持默认，模型内部已知是 en
        verbose=False, 
        fp16=use_fp16
    )
    
    text = result.get("text", "").strip()
    content = "".join(ch for ch in text if not ch.isspace())
    return len(content) >= min_chars

def main():
    # ... (原有主逻辑保持不变)
    folder = "test_indoor"
    
    if not os.path.isdir(folder):
        print(f"错误：未找到文件夹 '{folder}'")
        return

    wav_files = []
    for root, _, files in os.walk(folder):
        for fname in files:
            if fname.lower().endswith(".wav"):
                wav_files.append(os.path.join(root, fname))

    if not wav_files:
        print(f"没有找到 .wav 文件。")
        return

    total = len(wav_files)
    count_with_voice = 0
    no_voice_files = []

    print(f"开始检测（运行设备: {DEVICE}）: {folder}")
    
    for path in wav_files:
        name = os.path.relpath(path, folder)
        try:
            if has_voice_whisper(path):
                count_with_voice += 1
            else:
                no_voice_files.append(name)
        except Exception as e:
            print(f"无法处理文件 {name}: {e}")

    ratio = count_with_voice / total
    print("\n--- 检测结果 ---")
    print(f"包含语音数量: {count_with_voice} / {total} ({ratio:.2%})")

    if no_voice_files:
        print(f"\n未识别到语音的文件（{len(no_voice_files)}个）已列出。")

if __name__ == "__main__":
    main()

开始检测（运行设备: cuda）: test_indoor
正在加载英语专用模型至设备: cuda...


  0%|          | 0/1000 [00:00<?, ?frames/s]


--- 检测结果 ---
包含语音数量: 10 / 16 (62.50%)

未识别到语音的文件（6个）已列出。


In [1]:
import os
import torch
import torchaudio
import librosa  # 引入 librosa 作为备选后端，它更稳定
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor

/ihome/lshangguan/zhw227/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 1. 环境与模型配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name).to(device)
model.eval()

def classify_audio(file_path):
    """识别音频类别"""
    try:
        # 使用 librosa 兜底，确保兼容性
        y, sr = librosa.load(file_path, sr=16000)
        waveform = torch.from_numpy(y).unsqueeze(0) 

        inputs = feature_extractor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            
        probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
        _, top_indices = torch.topk(probabilities, k=5)
        
        id2label = model.config.id2label
        top_labels = [id2label[idx.item()] for idx in top_indices[0]]
        
        # 语音相关标签判定
        speech_related = ["Speech", "Conversation", "Babbling", "Child speech"]
        is_speech = any(any(s in label for s in speech_related) for label in top_labels)
        
        return "Speech" if is_speech else "Environment"
    except Exception as e:
        return f"Error: {e}"

# 2. 递归遍历 test 文件夹
input_root = "voice/test"  # 你的主文件夹名称
speech_list = []
env_list = []

print(f"开始扫描目录: {input_root} ...")

# os.walk 会自动进入每一层子文件夹
for root, dirs, files in os.walk(input_root):
    for filename in files:
        if filename.lower().endswith('.wav'):
            # 构建完整的文件路径
            file_path = os.path.join(root, filename)
            
            # 执行分类
            label = classify_audio(file_path)
            
            # 记录结果（包含相对路径，方便区分同名文件）
            relative_path = os.path.relpath(file_path, input_root)
            
            if label == "Speech":
                speech_list.append(relative_path)
            elif label == "Environment":
                env_list.append(relative_path)
            else:
                print(f"跳过错误文件: {relative_path} | 原因: {label}")

# 3. 输出汇总
print("\n" + "="*50)
print(f"扫描完成！在 '{input_root}' 及其子目录下共发现 {len(speech_list) + len(env_list)} 个有效音频。")
print("="*50)

print(f"\n[语音类文件 - 共 {len(speech_list)} 个]:")
for item in speech_list:
    print(f"  - {item}")

print(f"\n[环境音类文件 - 共 {len(env_list)} 个]:")
for item in env_list:
    print(f"  - {item}")

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 1964.28it/s, Materializing param=classifier.layernorm.weight]                                                    


开始扫描目录: voice/test ...

扫描完成！在 'voice/test' 及其子目录下共发现 5 个有效音频。

[语音类文件 - 共 5 个]:
  - 004.wav
  - 013.wav
  - 021.wav
  - 041.wav
  - 045.wav

[环境音类文件 - 共 0 个]:


In [1]:
import os
import torch
import librosa
import numpy as np
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor, pipeline

/ihome/lshangguan/zhw227/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# 1. 环境与模型初始化
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"检测到设备: {device} | 精度: {torch_dtype}")

# --- A. 加载分类模型 (AST) ---
ast_model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
ast_extractor = AutoFeatureExtractor.from_pretrained(ast_model_name)
ast_model = AutoModelForAudioClassification.from_pretrained(ast_model_name).to(device)
ast_model.eval()

# --- B. 加载转录模型 (Distil-Whisper) ---
distil_model_id = "distil-whisper/distil-large-v3"
print(f"正在加载转录模型...")

try:
    # 尝试使用加速方案加载
    transcribe_pipe = pipeline(
        "automatic-speech-recognition",
        model=distil_model_id,
        torch_dtype=torch_dtype,
        device=0 if device == "cuda" else -1,
        model_kwargs={"attn_implementation": "sdpa"} if device == "cuda" else {}
    )
except Exception as e:
    print(f"加速加载失败，尝试基础模式: {e}")
    # 最后的兜底方案
    transcribe_pipe = pipeline(
        "automatic-speech-recognition",
        model=distil_model_id,
        device=0 if device == "cuda" else -1
    )

def get_audio_label(file_path):
    """利用 AST 进行二分类"""
    try:
        y, sr = librosa.load(file_path, sr=16000)
        waveform = torch.from_numpy(y).unsqueeze(0) 
        inputs = ast_extractor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = ast_model(**inputs)
        
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
        _, top_indices = torch.topk(probs, k=5)
        id2label = ast_model.config.id2label
        top_labels = [id2label[idx.item()] for idx in top_indices[0]]
        
        speech_related = ["Speech", "Conversation", "Babbling", "Child speech"]
        is_speech = any(any(s in label for s in speech_related) for label in top_labels)
        return "Speech" if is_speech else "Environment"
    except Exception as e:
        return f"Error: {e}"

def transcribe_with_distil(file_path):
    """使用 Distil-Whisper 转录"""
    try:
        result = transcribe_pipe(file_path)
        return result["text"].strip()
    except Exception as e:
        return f"[转录失败: {e}]"

# 2. 执行循环 (与之前一致)
input_root = "indoor_public_hall"
results = []

print(f"\n开始处理目录: {input_root}")
for root, _, files in os.walk(input_root):
    for filename in files:
        if filename.lower().endswith('.wav'):
            file_path = os.path.join(root, filename)
            rel_path = os.path.relpath(file_path, input_root)
            label = get_audio_label(file_path)
            
            if label == "Speech":
                print(f"正在处理语音: {rel_path}")
                text = transcribe_with_distil(file_path)
                results.append({"file": rel_path, "type": "Speech", "content": text})
            else:
                results.append({"file": rel_path, "type": "Environment", "content": "-"})

# 3. 打印结果
print("\n" + "="*80)
print(f"{'文件名':<40} | {'类型':<12} | {'内容'}")
print("-" * 80)
for item in sorted(results, key=lambda x: x['file']):
    print(f"{item['file']:<40} | {item['type']:<12} | {item['content']}")

检测到设备: cuda | 精度: torch.float16


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 2174.70it/s, Materializing param=classifier.layernorm.weight]                                                    


正在加载转录模型...


Loading weights: 100%|██████████| 539/539 [00:00<00:00, 2237.69it/s, Materializing param=model.encoder.layers.31.self_attn_layer_norm.weight]



开始处理目录: indoor_public_hall
正在处理语音: A_baby_keeps_crying_1.wav
正在处理语音: A_baby_keeps_crying_2.wav
正在处理语音: A_baby_keeps_crying_3.wav
正在处理语音: A_baby_keeps_crying_4.wav
正在处理语音: A_baby_keeps_crying_5.wav
正在处理语音: A_nearby_crowd_suddenly_becomes_noisy_1.wav
正在处理语音: A_nearby_crowd_suddenly_becomes_noisy_2.wav
正在处理语音: A_nearby_crowd_suddenly_becomes_noisy_3.wav
正在处理语音: A_nearby_crowd_suddenly_becomes_noisy_4.wav
正在处理语音: A_nearby_crowd_suddenly_becomes_noisy_5.wav
正在处理语音: A_public_address_announcement_starts_playing_1.wav
正在处理语音: A_public_address_announcement_starts_playing_2.wav
正在处理语音: A_public_address_announcement_starts_playing_3.wav
正在处理语音: A_public_address_announcement_starts_playing_4.wav
正在处理语音: A_public_address_announcement_starts_playing_5.wav
正在处理语音: A_rolling_shutter_or_gate_closes_2.wav
正在处理语音: An_elevator_overload_or_fault_alarm_sounds_4.wav
正在处理语音: Cash_register_payment_success_tone_2.wav
正在处理语音: Cash_register_payment_success_tone_3.wav
正在处理语音: Cash_register_payment_success_tone_5.